In [4]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

1. **Data Understanding**

In [5]:
# 1. Load the dataset
df = pd.read_csv("IMDB Dataset.csv")

# 2. Explore number of samples and shape
print(f"Dataset Shape: {df.shape}")

# 3. Class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

# 4. Explore sample texts
print("\nSample Texts:")
print(df.head())

# Encode labels (Positive = 1, Negative = 0)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

Dataset Shape: (50000, 2)

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Texts:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


**2 . NLP Preprocessing**

In [6]:
# Initialize Lemmatizer and Stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Create a reusable preprocessing function
def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()

    # 2. Removing URLs/HTML tags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)

    # 3. Removing punctuation and special characters
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 4. Tokenization
    tokens = text.split()

    # 5. Removing stopwords and Lemmatization
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(cleaned_tokens)

# Apply preprocessing (Using a subset for faster execution during testing; remove [:5000] for full dataset)
print("Starting preprocessing...")
df['cleaned_review'] = df['review'][:5000].apply(preprocess_text)
df_subset = df.dropna(subset=['cleaned_review']) # Drop any NaNs from subsetting
print("Preprocessing complete. Sample cleaned text:")
print(df_subset['cleaned_review'].head())

Starting preprocessing...
Preprocessing complete. Sample cleaned text:
0    one reviewer mentioned watching oz episode you...
1    wonderful little production filming technique ...
2    thought wonderful way spend time hot summer we...
3    basically there family little boy jake think t...
4    petter matteis love time money visually stunni...
Name: cleaned_review, dtype: object


In [7]:
# Split the data before feature engineering to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    df_subset['cleaned_review'],
    df_subset['label'],
    test_size=0.2,
    random_state=42
)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Training samples: 4000, Testing samples: 1000


**3. Feature Engineering**

In [8]:
# 1. Bag of Words (BoW)
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# 2. TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Feature Engineering complete: BoW and TF-IDF vectors created.")

Feature Engineering complete: BoW and TF-IDF vectors created.


**4. Model Building**

In [10]:
# Initialize the three required models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

# Define a function to evaluate models
def train_and_evaluate(models, X_train, y_train, X_test, y_test, vectorization_method):
    results = []
    print(f"--- Results using {vectorization_method} ---")
    for name, model in models.items():
        # Train
        model.fit(X_train, y_train)
        # Predict
        y_pred = model.predict(X_test)

        # Evaluate [cite: 33]
        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')

        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        })
        print(f"{name} -> Accuracy: {acc:.4f} | F1: {f1:.4f}")
    return pd.DataFrame(results)

**5. Model Evaluation**

In [11]:
# Run evaluation for BoW
results_bow = train_and_evaluate(models, X_train_bow, y_train, X_test_bow, y_test, "Bag of Words")

# Run evaluation for TF-IDF
print("\n")
results_tfidf = train_and_evaluate(models, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF")

--- Results using Bag of Words ---
Logistic Regression -> Accuracy: 0.8550 | F1: 0.8501
Naive Bayes -> Accuracy: 0.8210 | F1: 0.8056
Decision Tree -> Accuracy: 0.6550 | F1: 0.6334


--- Results using TF-IDF ---
Logistic Regression -> Accuracy: 0.8630 | F1: 0.8571
Naive Bayes -> Accuracy: 0.8330 | F1: 0.8179
Decision Tree -> Accuracy: 0.6710 | F1: 0.6496


**6. Comparison & Insights**



Summary of Findings:

Best Preprocessing Steps:

Lowercasing and removing punctuation drastically reduced the noise in the dataset. Lemmatization was chosen over stemming because it reduces words to valid dictionary roots, which preserves semantic meaning better for sentiment analysis.

Best Vectorization (BoW vs. TF-IDF):

Generally, TF-IDF outperforms or matches BoW because it penalizes highly frequent words (like "movie", "film") that don't carry strong sentiment, while highlighting rare, meaningful words.

Best Model:

Logistic Regression typically emerges as the best performer among the three due to its ability to handle high-dimensional sparse data efficiently and find a linear decision boundary. Naive Bayes performs competitively and trains very quickly, making it a great baseline. Decision Trees usually overfit the training data and perform the worst on testing sets with high-dimensional text data.

Trade-offs:

Naive Bayes is fast and works well with word counts, but assumes independence between words.

Decision Trees are highly interpretable but prone to overfitting on sparse matrices.

Logistic Regression offers the best balance of accuracy and computational efficiency for this specific text classification task.